# Ch 32. 모니터링 · 피드백 루프 · 비용

운영의 마지막 관문: 품질 시그널 4가지(거절률·재요청률·환각·👍/👎), 드리프트 감지(KL divergence), 피드백 루프, 1년 비용 모델, 롤백 30초.

In [ ]:
# !pip install -q numpy

import math
import random
from collections import Counter

random.seed(42)
print("환경 준비 완료")

## 최소 예제 — 품질 시그널 4가지

### (1) 거절률 (Refusal rate)
모델이 "잘 모르겠습니다" 또는 답변 거부한 비율. 갑자기 상승하면 정렬 깨짐 또는 OOD 입력 증가 신호.

In [ ]:
def is_refusal(text):
    keywords = ["잘 모르겠", "도와드릴 수 없", "I don't know", "I cannot"]
    return any(k in text for k in keywords)

# 시뮬레이션 응답들
responses_normal = [
    "환불은 구매 후 7일 이내에 가능합니다.",
    "배송 예정일은 3~5 영업일입니다.",
    "잘 모르겠습니다.",
    "가격은 39,000원입니다.",
    "도와드릴 수 없는 요청입니다.",
]

refusal_rate = sum(is_refusal(r) for r in responses_normal) / len(responses_normal)
print(f"거절률: {refusal_rate:.1%}")

# 임계값 확인
threshold = 0.10  # 10% 초과 시 알림
if refusal_rate > threshold:
    print(f"경고: 거절률 {refusal_rate:.1%} > 임계 {threshold:.0%} — 모델 점검 필요")
else:
    print(f"거절률 정상 ({refusal_rate:.1%} <= {threshold:.0%})")

In [ ]:
# (3) 환각 신호 — 자기일관성
def self_consistency(model, q, k=5):
    """같은 질문 k번 → 답변 다르면 환각 의심."""
    # 실제로는 model 호출 k회 + pairwise BERT score
    # 여기서는 시뮬레이션
    answers = [f"답변 {i}" for i in range(k)]  # 실제: [run(model, q, temperature=0.7) for _ in range(k)]
    # similarities = [...]    # pairwise BERT score 등
    # return mean(similarities)    # 낮으면 환각 위험
    return 0.85  # 시뮬레이션

score = self_consistency(None, "환불 정책을 알려주세요")
print(f"자기일관성 점수: {score:.2f} {'→ 안전' if score > 0.7 else '→ 환각 위험'}")

## 최소 예제 — 드리프트 감지 (KL divergence)

학습 시 본 입력과 운영 입력의 분포 차이를 KL divergence 로 측정.

In [ ]:
def drift_kl(train_tokens, prod_tokens, vocab_size):
    """KL divergence — 토큰 빈도 비교."""
    train_counts = Counter(train_tokens)
    prod_counts = Counter(prod_tokens)
    train_total = sum(train_counts.values())
    prod_total = sum(prod_counts.values())
    kl = 0
    for tok in set(train_counts) | set(prod_counts):
        p = (prod_counts[tok] + 1) / (prod_total + vocab_size)
        q = (train_counts[tok] + 1) / (train_total + vocab_size)
        if p > 0:
            kl += p * math.log(p / q)
    return kl

# 시뮬레이션: 학습 토큰 분포 (동화 도메인)
vocab = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
train_tokens = random.choices(vocab[:15], k=10000)  # 학습 분포: A~O 위주

# 운영 입력 1: 유사 분포 (낮은 드리프트)
prod_tokens_low = random.choices(vocab[:18], k=1000)   # A~R 일부 추가
# 운영 입력 2: 다른 분포 (높은 드리프트)
prod_tokens_high = random.choices(vocab[10:], k=1000)  # K~Z 위주

kl_low = drift_kl(train_tokens, prod_tokens_low, len(vocab))
kl_high = drift_kl(train_tokens, prod_tokens_high, len(vocab))

print(f"낮은 드리프트 KL: {kl_low:.3f} {'→ 분포 일치' if kl_low < 0.1 else '→ 살짝 시프트' if kl_low < 0.5 else '→ 드리프트'}")
print(f"높은 드리프트 KL: {kl_high:.3f} {'→ 분포 일치' if kl_high < 0.1 else '→ 살짝 시프트' if kl_high < 0.5 else '→ 드리프트 — 재학습 검토'}")

## 실전 — 1년 비용 모델

자체 모델 vs API 비용 비교.

In [ ]:
# 본 책 캡스톤 모델 (Qwen 0.5B + LoRA) 운영 비용 (가설)
cost_self = {
    "GPU 서빙 (T4)": 200,             # 월
    "분기 재학습 (Colab Pro)": 50/3,  # 월 (분기당 $50)
    "라벨링·합성 (Haiku)": 50,
    "라이선스": 0,                    # Apache
    "모니터링 인프라": 50,
}
monthly_self = sum(cost_self.values())
yearly_self = monthly_self * 12

# API 비교 (Claude Sonnet)
calls_per_month = 100_000
avg_tokens = 1500
price_per_M = 3.0
monthly_api = calls_per_month * avg_tokens / 1_000_000 * price_per_M
yearly_api = monthly_api * 12

print("=== 자체 모델 비용 ===")
for k, v in cost_self.items():
    print(f"  {k:<28} ${v:>6.0f}/월")
print(f"  {'합계':<28} ${monthly_self:>6.0f}/월 = ${yearly_self:,.0f}/년")

print()
print(f"=== API (Claude Sonnet) 비교 ===")
print(f"  콜 {calls_per_month:,}건/월 × {avg_tokens} 토큰 × ${price_per_M}/M")
print(f"  ${monthly_api:>6.0f}/월 = ${yearly_api:,.0f}/년")

savings = yearly_api - yearly_self
print()
print(f"절약: ${savings:,.0f}/년 {'(자체 모델 유리)' if savings > 0 else '(API 유리)'}")
print(f"ROI 양수 조건: 트래픽 증가, PII (외부 불가), 또는 latency 100ms 요건")

## 실전 — 분기 운영 사이클 시뮬레이션

In [ ]:
# 롤백 전략
# 어댑터 v2 → v1 롤백
# client.set_lora("adapter_v1")        # vLLM API
# 또는 다른 어댑터로 전체 라우팅

rollback_times = {
    "어댑터 swap (LoRA)": "30초",
    "컨테이너 재시작":    "1분",
    "베이스 모델 교체":   "10분",
}

print("롤백 전략별 소요 시간:")
for strategy, t in rollback_times.items():
    print(f"  {strategy:<20} → {t}")

print()
print("분기 운영 사이클 (3개월):")
cycle = [
    ("1주",    "신모델 카나리 (Ch 30) — 5% 트래픽"),
    ("2주",    "A/B 100% ramp"),
    ("1개월",  "모니터링 + 피드백 수집"),
    ("2개월",  "거절·재요청 케이스 라벨링 + IAA"),
    ("3개월",  "새 학습 데이터 + 학습 + 평가"),
    ("다음 분기 1주", "신모델 카나리"),
]
for t, action in cycle:
    print(f"  {t:<15} → {action}")

## 연습문제

1. 본 책 모델 운영 가설로 1주일 로그 시뮬레이션. 4 시그널 대시보드 작성.
2. KL divergence 측정 — 학습 데이터 vs 시뮬레이션 운영 입력.
3. 피드백 루프 자동화 — 거절 케이스 100건을 다음 학습 데이터에 합류.
4. 본인 도메인의 1년 비용 모델 작성 (자체 vs API).
5. **(생각해볼 것)** "분기마다 재학습" 이 항상 옳은가? 학습 안 해야 할 이유?

In [ ]:
# 연습 1: 1주일 운영 로그 시뮬레이션 + 4 시그널 대시보드


In [ ]:
# 연습 2: KL divergence — 학습 데이터 토큰 분포 vs 운영 입력 토큰 분포


In [ ]:
# 연습 3: 피드백 루프 자동화 — 거절 케이스 추출 + 다음 학습 데이터 합류


In [ ]:
# 연습 4: 본인 도메인 1년 비용 모델 (자체 모델 vs API)
